# Airfoil Self-Noise Trainer and Predictor: Workflow
Use this notebook to predict the **scaled sound pressure level (dB)** of a NACA 0012 airfoil for **your own data**, with the final model of the report (neural network regression with scaled MLP and one hidden layer of 50 neurons).

### What you need
A CSV file with a header row and these five columns:

| Column | Unit |
| --- | --- |
| `Frequency_Hz` | Hz |
| `Angle_of_attack_deg` | degrees |
| `Chord_length_m` | m |
| `Free_stream_velocity_m/s` | m/s |
| `Suction_side_displacement_thickness_m` | m |

An example is in `../data/example_input.csv`. If your file also has the column `Scaled_sound_pressure_level_dB` (measured values), the notebook compares the predictions with them.

### How to use
1. Put your file in the `data/` folder and change `INPUT_FILE` in step 1.
2. Run **Kernel > Restart and Run All**.
3. The predictions are saved to `OUTPUT_FILE`.

### How invalid input is handled
* Missing data, non-number values, missing (NaN), repeated data, or negative frequency, chord length, velocity and/or thickness: the notebook stops with a clear error message.
* Inputs outside the range the model was trained on: the prediction is still made, but the row is marked `Outside_training_range = True`. Treat these predictions with caution, because the model has never seen such values.

### 1. Settings

In [1]:
INPUT_FILE = "../data/example_input.csv"      # replace your file for example_input.csv (CSV with the five input columns)
OUTPUT_FILE = "../results/predictions.csv"    # where the predictions are saved

## 2. Train the final model
The model is trained on the NASA airfoil data (the 80% train + validation part). Its accuracy on the untouched 20% test set.

In [2]:
# import required modules
import os

import matplotlib.pyplot as plt

from airfoil_utils import TARGET_COLUMN, regression_metrics
from workflow import (FLAG_COLUMN, PREDICTION_COLUMN, predict_new_data,
                      read_user_data, train_reference_model)

# file path access to "airfoil_self_noise.dat" for training to obtain the trained model, training inputs and validation test metrics
model, reference_inputs, test_metrics = train_reference_model()

# print the model
print("Final model:", model.steps[-1][1].hidden_layer_sizes, "hidden neurons")
# print the validation test metrics
print(f"Accuracy on the reference test set: MAE {test_metrics['MAE']:.2f} dB | "
      f"RMSE {test_metrics['RMSE']:.2f} dB | R2 {test_metrics['R2']:.2f}")

Final model: (50,) hidden neurons
Accuracy on the reference test set: MAE 3.04 dB | RMSE 3.96 dB | R2 0.68


## 3. Read and check your data
`read_user_data` checks that the file exists and has the five required columns. The values themselves (numbers, no missing values, no negatives) are checked in the next step.

In [3]:
# access and read the user's input file
user_df = read_user_data(INPUT_FILE)
# print the first 5 rows of the user's input file
print(f"{len(user_df)} rows read from {INPUT_FILE}")
user_df.head()

5 rows read from ../data/example_input.csv


,Frequency_Hz,Angle_of_attack_deg,Chord_length_m,Free_stream_velocity_m/s,Suction_side_displacement_thickness_m
0,800,0.0,0.3048,71.3,0.002663
1,1000,0.0,0.3048,71.3,0.002663
2,1250,0.0,0.3048,71.3,0.002663
3,1600,0.0,0.3048,71.3,0.002663
4,2000,0.0,0.3048,71.3,0.002663


## 4. Predict
`predict_new_data` validates every value, predicts the sound pressure level, and marks rows that are outside the range of the training data.

In [4]:
# the function takes in the trained model, users input features and training inputs
result = predict_new_data(model, user_df, reference_inputs)

# from the training inputs, check extrapolation
n_outside = int(result[FLAG_COLUMN].sum())
if n_outside:
    print(f"WARNING: {n_outside} row(s) are outside the training range. Treat them with caution.")
else:
    print("All rows are inside the range of the training data.")

result

All rows are inside the range of the training data.


,Frequency_Hz,Angle_of_attack_deg,Chord_length_m,Free_stream_velocity_m/s,Suction_side_displacement_thickness_m,Predicted_SPL_dB,Outside_training_range
0,800,0.0,0.3048,71.3,0.002663,132.174260,False
1,1000,0.0,0.3048,71.3,0.002663,131.365624,False
2,1250,0.0,0.3048,71.3,0.002663,130.354828,False
3,1600,0.0,0.3048,71.3,0.002663,129.273458,False
4,2000,0.0,0.3048,71.3,0.002663,128.077931,False


## 5. Save the predictions

In [5]:
# auto save the predicted values via the file path
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
result.to_csv(OUTPUT_FILE, index=False)
print(f"{len(result)} predictions saved to {OUTPUT_FILE}")

5 predictions saved to ../results/predictions.csv


## 6. Optional: compare with measured values
This step only runs if your file contains the measured column `Scaled_sound_pressure_level_dB`.

*Note: the accuracy is a fair check only for data the model has not seen. The example file is taken from the training data, so its accuracy looks better than it would for new measurements.*

In [6]:
# Calculates the validation metrcis of the predicted SPL and actual SPL (provided by users)
if TARGET_COLUMN in result.columns:
    scores = regression_metrics(result[TARGET_COLUMN], result[PREDICTION_COLUMN])
    print(f"MAE {scores['MAE']:.2f} dB | RMSE {scores['RMSE']:.2f} dB | R2 {scores['R2']:.2f}")

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(result[TARGET_COLUMN], result[PREDICTION_COLUMN])
    limits = [result[TARGET_COLUMN].min(), result[TARGET_COLUMN].max()]
    ax.plot(limits, limits, "r--", label="Perfect prediction")
    ax.set_xlabel("Measured SPL (dB)")
    ax.set_ylabel("Predicted SPL (dB)")
    ax.set_title("Predicted vs measured")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No measured column in the file, so there is nothing to compare.")

No measured column in the file, so there is nothing to compare.
